In [0]:
df = spark.sql("""
SELECT p.*
FROM policyprojcatalog.policyprojdb.policy p
INNER JOIN policyprojcatalog.policyprojdb.customer c
  ON p.customer_id = c.customer_id
WHERE p.customer_id IS NOT NULL
  AND p.policy_id IS NOT NULL
  AND p.merge_flag = false
  AND p.premium > 0
  AND p.coverage_amount > 0
  AND p.end_date > p.start_date
""")

display(df)

In [0]:
df.createOrReplaceTempView("clean_policy")

spark.sql("""
MERGE INTO policyprojcatalog.silver.policy AS T
USING clean_policy AS S
ON T.policy_id = S.policy_id

WHEN MATCHED THEN UPDATE SET
  T.policy_type = S.policy_type,
  T.premium = S.premium,
  T.end_date = S.end_date,
  T.start_date = S.start_date,
  T.coverage_amount = S.coverage_amount,
  T.customer_id = S.customer_id,
  T.merged_timestamp = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
  policy_id,
  policy_type,
  premium,
  end_date,
  start_date,
  coverage_amount,
  customer_id,
  merged_timestamp
)
VALUES (
  S.policy_id,
  S.policy_type,
  S.premium,
  S.end_date,
  S.start_date,
  S.coverage_amount,
  S.customer_id,
  current_timestamp()
)
""")

In [0]:
spark.sql("""
UPDATE policyprojcatalog.policyprojdb.policy
SET merge_flag = true
WHERE merge_flag = false
""")